# HopLab — Motor de análisis

**Uso:** `Runtime → Run all` (Ctrl+F9)  
Al terminar, la última celda imprimirá la URL pública del motor.  
Pégala en la UI (Vercel) en el campo **Conectar motor**.

> Asegúrate de que el runtime tiene **GPU** activa:  
> `Runtime → Change runtime type → T4 GPU`

---
## SOLO OWNER: crear estructura Drive

Ejecuta **una sola vez** (antes de invitar invitados). Crea `MyDrive/hoplab-data` + subcarpetas e imprime el **folder ID** para `HOPLAB_DATA_FOLDER_ID`.

Los invitados **no** deben ejecutar esta celda. En runs posteriores del owner puedes omitirla.


In [ ]:
# ─── SOLO OWNER: crear estructura Drive (ejecutar UNA vez; omitir después) ───
# Los invitados NO deben ejecutar esta celda.
# Copia equivalente: hoplab-cloud/colab/owner_bootstrap_drive.py

from __future__ import annotations

import pathlib

from google.colab import auth, drive
from googleapiclient.discovery import build

FOLDER_NAME = "hoplab-data"
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive") / FOLDER_NAME
SUBDIRS = ("videos", "output", "venues", "models", "logs")
_FOLDER_MIME = "application/vnd.google-apps.folder"


def _ensure_local_tree() -> None:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    for name in SUBDIRS:
        (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
    print(f"✅ Estructura local lista: {DRIVE_ROOT}")
    for name in SUBDIRS:
        print(f"   · {name}/")


def _find_or_create_folder_id(service) -> str:
    """Busca `hoplab-data` bajo My Drive (root); la crea si no existe."""
    query = (
        f"name = '{FOLDER_NAME}' and mimeType = '{_FOLDER_MIME}' "
        "and 'root' in parents and trashed = false"
    )
    resp = (
        service.files()
        .list(q=query, spaces="drive", fields="files(id, name)", pageSize=10)
        .execute()
    )
    files = resp.get("files", [])
    if files:
        return files[0]["id"]

    meta = {
        "name": FOLDER_NAME,
        "mimeType": _FOLDER_MIME,
        "parents": ["root"],
    }
    created = service.files().create(body=meta, fields="id").execute()
    print(f"📁 Carpeta '{FOLDER_NAME}' creada en My Drive vía API.")
    return created["id"]


def _find_child_folder(service, parent_id: str, name: str):
    query = (
        f"name = '{name}' and mimeType = '{_FOLDER_MIME}' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    resp = (
        service.files()
        .list(q=query, spaces="drive", fields="files(id, name)", pageSize=10)
        .execute()
    )
    files = resp.get("files", [])
    return files[0]["id"] if files else None


def _ensure_subdirs_via_api(service, parent_id: str) -> None:
    for name in SUBDIRS:
        existing = _find_child_folder(service, parent_id, name)
        if existing:
            print(f"   · API: '{name}/' ya existe (id={existing})")
            continue
        meta = {
            "name": name,
            "mimeType": _FOLDER_MIME,
            "parents": [parent_id],
        }
        created = service.files().create(body=meta, fields="id").execute()
        print(f"   · API: creada '{name}/' (id={created['id']})")


print("🔐 Montando Google Drive (MyDrive)…")
drive.mount("/content/drive")

_ensure_local_tree()

print("🔑 Autenticando para Drive API v3…")
auth.authenticate_user()
service = build("drive", "v3")

folder_id = _find_or_create_folder_id(service)
print(f"📁 Asegurando subcarpetas vía API bajo {folder_id}…")
_ensure_subdirs_via_api(service, folder_id)
share_url = f"https://drive.google.com/drive/folders/{folder_id}"

print()
print("─── RESULTADO (owner) ───────────────────────────────────────────")
print(f"Ruta:       {DRIVE_ROOT}")
print(f"Folder ID:  {folder_id}")
print(f"URL share:  {share_url}")
print()
print("Comparte esa carpeta con los invitados (Editor) y dales:")
print(f'HOPLAB_DATA_FOLDER_ID = "{folder_id}"')
print("─────────────────────────────────────────────────────────────────")


In [ ]:
# ─── CELDA 0 — CONFIGURACIÓN (única que debes editar) ────────────────────────
# Repo del engine (cámbialo cuando lo subas a GitHub)
REPO_URL    = "https://github.com/TU-USUARIO/TU-REPO.git"
REPO_BRANCH = "main"

# Google Drive: True = persistencia entre sesiones (recomendado)
USE_DRIVE   = True
DRIVE_DATA  = "/content/drive/MyDrive/hoplab-data"  # owner: carpeta propia en MyDrive

# Owner: deja vacío → MyDrive/hoplab-data (mkdir local basta).
#   Si pegas tu propio folder ID (p.ej. para probar la ruta compartida), las
#   subcarpetas se crean vía Drive API si FUSE mkdir falla (errno 95).
# Invitado: pega el ID de la carpeta compartida `hoplab-data` del OWNER.
# ID = segmento de la URL: https://drive.google.com/drive/folders/FOLDER_ID
# También puedes exportar HOPLAB_DATA_FOLDER_ID en el entorno antes de Run All.
HOPLAB_DATA_FOLDER_ID = ""

# Raíz alternativa (invitado): más fiable para escrituras que .shortcut-targets-by-id.
# Vacío = auto (ver orden en celda de Drive). Tras "Añadir acceso directo a Mi unidad"
# en la carpeta compartida, suele funcionar mejor:
#   HOPLAB_DATA_ROOT = "/content/drive/MyDrive/hoplab-data"
HOPLAB_DATA_ROOT = ""

# Puerto de la API
API_PORT    = 8000
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── CELDA 1 — VERIFICAR GPU ─────────────────────────────────────────────────
import subprocess, sys
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ GPU detectada:", result.stdout.strip())
    else:
        print("⚠️  Sin GPU — el análisis será más lento (CPU).")
        print("   Ir a: Runtime → Change runtime type → T4 GPU")
except FileNotFoundError:
    print("⚠️  nvidia-smi no disponible — sin GPU.")

# Verificar PyTorch (ya instalado en Colab)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch {torch.__version__} | CUDA disponible: {cuda_ok}")
except ImportError:
    print("PyTorch no encontrado — se instalará en la siguiente celda.")

In [ ]:
# ─── CELDA 2 — MONTAR DRIVE y CREAR ESTRUCTURA DE CARPETAS ───────────────────
import errno
import os
import pathlib
import time

_REQUIRED_SUBDIRS = ("videos", "output", "venues", "models", "logs")
_FOLDER_MIME = "application/vnd.google-apps.folder"

_ERR_MKDIR_SHARED = (
    "Google Drive FUSE no permite mkdir en esta ruta (errno 95 / EOPNOTSUPP), "
    "típico de carpetas vía `.shortcut-targets-by-id`.\n\n"
    "Con HOPLAB_DATA_FOLDER_ID definido, esta celda crea las subcarpetas vía "
    "Drive API (requiere auth.authenticate_user y rol Editor).\n"
    "Si aún falla: Organizar → «Añadir acceso directo a Mi unidad» y pon:\n"
    "  HOPLAB_DATA_ROOT = \"/content/drive/MyDrive/hoplab-data\""
)


def ensure_dir(path: pathlib.Path) -> bool:
    """Intenta mkdir local. True si el dir existe; False si errno 95 y aún falta."""
    path = pathlib.Path(path)
    try:
        path.mkdir(parents=True, exist_ok=True)
        return True
    except OSError as e:
        if e.errno not in (errno.EOPNOTSUPP, 95):
            raise
        if path.exists() and path.is_dir():
            return True
        return False


def _drive_service():
    from google.colab import auth
    from googleapiclient.discovery import build
    print("🔑 Autenticando Drive API v3 (puede pedir consent)…")
    auth.authenticate_user()
    return build("drive", "v3")


def _find_child_folder(service, parent_id: str, name: str):
    query = (
        f"name = '{name}' and mimeType = '{_FOLDER_MIME}' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    resp = (
        service.files()
        .list(q=query, spaces="drive", fields="files(id, name)", pageSize=10)
        .execute()
    )
    files = resp.get("files", [])
    return files[0]["id"] if files else None


def _ensure_subdir_via_api(service, parent_id: str, name: str) -> str:
    existing = _find_child_folder(service, parent_id, name)
    if existing:
        print(f"   · API: '{name}/' ya existe (id={existing})")
        return existing
    meta = {
        "name": name,
        "mimeType": _FOLDER_MIME,
        "parents": [parent_id],
    }
    created = service.files().create(body=meta, fields="id").execute()
    print(f"   · API: creada '{name}/' (id={created['id']})")
    return created["id"]


def _wait_fuse_subdir(root: pathlib.Path, name: str, timeout_s: float = 15.0) -> bool:
    target = root / name
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if target.is_dir():
            return True
        time.sleep(1.0)
    return target.is_dir()


def ensure_required_subdirs(data_root: pathlib.Path) -> None:
    """mkdir local primero; si FUSE falla y hay folder ID → Drive API v3."""
    folder_id = (HOPLAB_DATA_FOLDER_ID or os.environ.get("HOPLAB_DATA_FOLDER_ID", "")).strip()
    missing = []
    for sub in _REQUIRED_SUBDIRS:
        if not ensure_dir(data_root / sub):
            missing.append(sub)

    if not missing:
        return

    if not folder_id:
        raise OSError(
            errno.EOPNOTSUPP,
            f"{_ERR_MKDIR_SHARED}\n\n"
            f"Carpetas que faltan bajo {data_root}: {', '.join(missing)}\n"
            "(Sin HOPLAB_DATA_FOLDER_ID no hay fallback API; owner deja ID vacío "
            "y usa MyDrive, o define el folder ID.)",
        )

    print(
        f"⚠️  mkdir FUSE falló para: {', '.join(missing)}\n"
        f"   Creando vía Drive API bajo parent {folder_id}…"
    )
    service = _drive_service()
    for sub in missing:
        _ensure_subdir_via_api(service, folder_id, sub)

    still_missing = []
    for sub in missing:
        if _wait_fuse_subdir(data_root, sub):
            print(f"   ✓ FUSE ve {data_root / sub}")
        else:
            still_missing.append(sub)

    if still_missing:
        print(
            "ℹ️  Creadas en Drive API, pero FUSE aún no las muestra:\n"
            f"   {', '.join(still_missing)} bajo {data_root}\n"
            "   Re-ejecuta esta celda en unos segundos (lag de montaje)."
        )


def _resolve_data_root() -> pathlib.Path:
    """Orden: HOPLAB_DATA_ROOT → folder ID → MyDrive/hoplab-data."""
    explicit = (HOPLAB_DATA_ROOT or os.environ.get("HOPLAB_DATA_ROOT", "")).strip()
    if explicit:
        root = pathlib.Path(explicit)
        if not root.exists():
            raise FileNotFoundError(
                f"HOPLAB_DATA_ROOT no existe: {root}\n"
                "Comprueba el acceso directo en Mi unidad o corrige la ruta."
            )
        print(f"📂 Raíz explícita (HOPLAB_DATA_ROOT). DATA_ROOT = {root}")
        return root

    folder_id = (HOPLAB_DATA_FOLDER_ID or os.environ.get("HOPLAB_DATA_FOLDER_ID", "")).strip()
    if folder_id:
        shared = pathlib.Path("/content/drive/.shortcut-targets-by-id") / folder_id
        if shared.exists():
            print(
                f"📂 Carpeta compartida por ID. DATA_ROOT = {shared}\n"
                "   Nota: si mkdir FUSE falla (errno 95), se crean subcarpetas "
                "vía Drive API automáticamente."
            )
            return shared
        fallback = pathlib.Path(DRIVE_DATA)
        print(
            f"⚠️  No se encontró la carpeta compartida:\n"
            f"   {shared}\n"
            f"   Comprueba: (1) aceptaste la invitación de Drive del OWNER,\n"
            f"   (2) el ID es el de la carpeta `hoplab-data` (no un padre),\n"
            f"   (3) espera unos segundos tras montar Drive y vuelve a ejecutar esta celda.\n"
            f"   Intentando fallback: {fallback}"
        )
        if fallback.exists():
            print(f"📂 Fallback MyDrive. DATA_ROOT = {fallback}")
            return fallback
        raise FileNotFoundError(
            f"No existe la carpeta compartida ({shared}) ni el fallback ({fallback}). "
            "Acepta el share, verifica HOPLAB_DATA_FOLDER_ID y re-ejecuta tras montar Drive."
        )
    root = pathlib.Path(DRIVE_DATA)
    print(f"📂 Drive montado (MyDrive). DATA_ROOT = {root}")
    return root


if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = _resolve_data_root()
else:
    DATA_ROOT = pathlib.Path("/content/hoplab-data")
    print(f"📂 Usando almacenamiento efímero. DATA_ROOT = {DATA_ROOT}")

ensure_required_subdirs(DATA_ROOT)

print("✅ Estructura de carpetas lista:")
for sub in ["videos", "output", "venues", "models"]:
    p = DATA_ROOT / sub
    n = sum(1 for _ in p.iterdir()) if p.exists() else 0
    print(f"   {p}  ({n} elementos)")


In [ ]:
# ─── CELDA 3 — CLONAR / ACTUALIZAR ENGINE (efímero en /content) ─────────────
import subprocess, pathlib

ENGINE_DIR = pathlib.Path("/content/hoplab-engine")

if ENGINE_DIR.exists():
    print("Actualizando engine existente...")
    r = subprocess.run(["git", "-C", str(ENGINE_DIR), "pull", "--ff-only"],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print(f"Clonando {REPO_URL} (rama {REPO_BRANCH})...")
    r = subprocess.run(["git", "clone", "--branch", REPO_BRANCH,
                        "--depth", "1", REPO_URL, str(ENGINE_DIR)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Error clonando repo: {r.stderr}")

print(f"✅ Engine listo en {ENGINE_DIR}")

In [ ]:
# ─── CELDA 4 — INSTALAR DEPENDENCIAS ─────────────────────────────────────────
# PyTorch ya viene en Colab con CUDA. Solo instalamos el resto.
import subprocess, sys

REQS = ENGINE_DIR / "hoplab-cloud" / "colab" / "requirements-colab.txt"
if not REQS.exists():
    # Fallback: instalar directamente
    pkgs = ["ultralytics>=8.3", "opencv-python-headless>=4.9",
            "numpy>=1.26", "scipy>=1.13", "matplotlib>=3.9",
            "fastapi>=0.115", "uvicorn>=0.30", "python-multipart>=0.0.9"]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs,
                       capture_output=True, text=True)
else:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "-r", str(REQS)], capture_output=True, text=True)

if r.returncode != 0:
    print("[ERROR pip]:", r.stderr[-2000:])
else:
    print("✅ Dependencias instaladas")

# Cachear pesos YOLO en Drive/models para no descargar cada sesión
import os
os.environ["YOLO_CONFIG_DIR"] = str(DATA_ROOT / "models")
print(f"   Pesos YOLO se cachearán en: {DATA_ROOT / 'models'}")

In [ ]:
# ─── CELDA 5 — CONFIGURAR ENV + SYMLINKS Drive ←→ Engine ─────────────────────
import os, pathlib, sys

# Vars de entorno que api_server.py lee
os.environ["HOPLAB_OUTPUT_ROOT"] = str(DATA_ROOT / "output")
os.environ["HOPLAB_VENUE_ROOT"]  = str(DATA_ROOT / "venues")
os.environ["HOPLAB_VIDEO_ROOT"]  = str(DATA_ROOT / "videos")
os.environ["YOLO_CONFIG_DIR"]    = str(DATA_ROOT / "models")

# Optimización: no escribir frames anotados a Drive (ahorra cuota)
os.environ["TJ_WRITE_ANNOTATED"] = "0"

# Cambiar al directorio del engine para que los imports relativos funcionen
os.chdir(str(ENGINE_DIR))
if str(ENGINE_DIR) not in sys.path:
    sys.path.insert(0, str(ENGINE_DIR))

# Symlinks para compatibilidad con paths que usa el engine directamente.
# El clone del repo puede traer dirs reales (venues/, output/) que bloquean unlink().
def safe_symlink(src: pathlib.Path, dst: pathlib.Path):
    """Apunta dst → src. Maneja symlink previo, archivo o directorio real."""
    src = pathlib.Path(src)
    dst = pathlib.Path(dst)
    if not ensure_dir(src):
        raise OSError(
            errno.EOPNOTSUPP,
            f"No existe {src}. Re-ejecuta la celda de Drive "
            "(fallback API) o crea la subcarpeta en Drive web.",
        )

    if dst.is_symlink():
        if dst.resolve() == src.resolve():
            print(f"   ✓ {dst} ya apunta a {src}")
            return
        dst.unlink()
    elif dst.exists():
        if dst.is_dir():
            if any(dst.iterdir()):
                # Preservar contenido local (p.ej. venues/ del repo) y liberar el path
                aside = dst.with_name(dst.name + ".local.bak")
                n = 1
                while aside.exists():
                    aside = dst.with_name(f"{dst.name}.local.bak.{n}")
                    n += 1
                dst.rename(aside)
                print(f"   ⚠ {dst} era un directorio con contenido → movido a {aside.name}")
            else:
                dst.rmdir()
        else:
            dst.unlink()

    dst.symlink_to(src, target_is_directory=True)
    print(f"   🔗 {dst} → {src}")

print("Creando symlinks...")
safe_symlink(DATA_ROOT / "output", ENGINE_DIR / "output")
safe_symlink(DATA_ROOT / "venues", ENGINE_DIR / "venues")

print("✅ Entorno configurado:")
for k in ["HOPLAB_OUTPUT_ROOT", "HOPLAB_VENUE_ROOT", "HOPLAB_VIDEO_ROOT",
          "TJ_WRITE_ANNOTATED", "YOLO_CONFIG_DIR"]:
    print(f"   {k} = {os.environ[k]}")

In [ ]:
# ─── CELDA 6 — ARRANCAR API ───────────────────────────────────────────────────
import threading, time, urllib.request, urllib.error

def _run_server():
    import uvicorn
    import api_server  # cwd = ENGINE_DIR
    uvicorn.run(api_server.app, host="0.0.0.0", port=API_PORT,
                log_level="warning")

srv_thread = threading.Thread(target=_run_server, daemon=True)
srv_thread.start()

print(f"Esperando API en :{API_PORT} ", end="")
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/status", timeout=2)
        print(" ✅ API lista")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
else:
    raise RuntimeError("API no respondió en 120s — revisa errores arriba")

In [ ]:
# ─── CELDA 7 — TUNNEL CLOUDFLARED (gratis, sin cuenta) ───────────────────────
import subprocess, threading, re, time, pathlib

CF_DEB = pathlib.Path("/tmp/cloudflared.deb")

if not pathlib.Path("/usr/bin/cloudflared").exists():
    print("Instalando cloudflared...")
    subprocess.run(
        ["wget", "-q", "-O", str(CF_DEB),
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],
        check=True)
    subprocess.run(["dpkg", "-i", str(CF_DEB)], check=True, capture_output=True)
    print("Cloudflared instalado.")

# Asegurarse de que no hay config.yaml que bloquee los quick tunnels
cf_cfg = pathlib.Path.home() / ".cloudflared" / "config.yaml"
if cf_cfg.exists():
    cf_cfg.rename(cf_cfg.with_suffix(".yaml.bak"))

TUNNEL_URL = None
tunnel_ready = threading.Event()

def _run_tunnel():
    global TUNNEL_URL
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{API_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True)
    pattern = re.compile(r"https://[\w-]+\.trycloudflare\.com")
    for line in proc.stdout:
        m = pattern.search(line)
        if m:
            TUNNEL_URL = m.group(0)
            tunnel_ready.set()

t = threading.Thread(target=_run_tunnel, daemon=True)
t.start()

print("Esperando URL del tunnel (máx 60s) ", end="")
ok = tunnel_ready.wait(timeout=60)
if not ok:
    print("\n⚠️  cloudflared no imprimió URL — intenta la celda de fallback debajo")
else:
    print(f"\n✅ Tunnel listo")

In [ ]:
# ─── CELDA 8 — URL PÚBLICA + INSTRUCCIONES ───────────────────────────────────
from IPython.display import display, HTML

if TUNNEL_URL:
    display(HTML(f"""
    <div style="font-family:monospace;background:#1e1e2e;color:#cdd6f4;
                padding:20px;border-radius:12px;font-size:15px;margin:10px 0">
      <b style="font-size:18px;color:#a6e3a1">✅ Motor HopLab listo</b><br><br>

      <b>URL del motor (copiar):</b><br>
      <span style="background:#313244;padding:6px 12px;border-radius:6px;
                   color:#89b4fa;font-size:16px;display:inline-block;margin:8px 0">
        {TUNNEL_URL}
      </span><br><br>

      <b>Siguiente paso:</b><br>
      1. Abre <a href='https://hoplab.vercel.app' style='color:#89dceb'>
            hoplab.vercel.app</a> (o tu URL de Vercel)<br>
      2. Haz clic en <b>Conectar motor</b><br>
      3. Pega la URL de arriba y guarda<br><br>

      <small style='color:#6c7086'>La URL cambia cada sesión de Colab.<br>
      Los análisis se guardan en Drive aunque cierres esta pestaña.</small>
    </div>
    """))
    print(f"\nURL: {TUNNEL_URL}")
    print(f"Docs: {TUNNEL_URL}/docs")
else:
    print("⚠️  Tunnel no disponible. Usa la celda de fallback (localtunnel) abajo.")

In [ ]:
# ─── CELDA 9 — FALLBACK: localtunnel (si cloudflared falla) ─────────────────
# Solo ejecutar si la celda 7 no imprimió URL.
import subprocess, sys

subprocess.run(["npm", "install", "-g", "localtunnel", "--silent"],
               capture_output=True)

lt_proc = subprocess.Popen(
    ["lt", "--port", str(API_PORT)],
    stdout=subprocess.PIPE, text=True)

for line in lt_proc.stdout:
    if "loca.lt" in line:
        lt_url = line.strip().split()[-1]
        print(f"\nFallback URL (localtunnel): {lt_url}")
        print("Nota: localtunnel puede pedir confirmación al abrir en el navegador.")
        break